In [1]:
# GLOBAL #

import random
import pandas as pd
from collections import defaultdict, Counter
import numpy as np
from itertools import product
from sklearn.model_selection import train_test_split

random.seed(123)
np.random.seed(123)

In [ ]:
# HELPER FUNCTIONS #

# TODO: double-check that readSequences joining parts is correct

#create strings from the .csv file
def readSequences(species_df, by_length=True):

        sequences = []

        grouped = species_df.groupby(
            ["trial_id", "sequence_id", "fish_id"]
        )

        for (_, _, fish_id), group in grouped:

            group = group.sort_values("order_id")
            parts = []

            for _, row in group.iterrows():

                if by_length:
                    parts.append(str(row["state"]) * int(row["length"]))
                else:
                    parts.append(str(row["state"]))

            sequences.append("".join(parts) + "\n")

        return sequences

# collapse sequence by self-transitions (e.g. [aaaabbbbcccdddccc] = [abcdc]) for collapsed context model
def collapseLength(seq):

    collapsed = []

    for s in seq:
        if not collapsed or collapsed[-1] != s:
            collapsed.append(s)

    return collapsed

# check if tuple has consecutive repeated states (we want to ignore these combinations in collapsed context model transition matrix)
def hasConsecutiveRepeats(states):

    for i in range(len(states) - 1):
        if states[i] == states[i + 1]:
            return True
        
    return False

# find indices of first state in each state run within sequence (e.g., [aaaabbbbcccdddccccc] = [0, 4, 8, 11, 14])
def uniqueIdxs(seq):

    lasts = []
    runChar = seq[0]

    for i, s in enumerate(seq[1:], start=1):
        if s != runChar:
            lasts.append(i - 1)
            runChar = s

    lasts.append(len(seq) - 1)
    return lasts

# expand  numeric vector into repetitions corresponding to each number value (e.g., [0, 4, 8, 11, 14] = [0, 0, 0, 0, 4, 4, 4, 4, 8, 8, 8, 11, 11, 11, 14, 14, 14]) 
# for matching strings to indices in collapsed context model 
def expandVec(vec):

    if not vec:
        return []
    
    result = []

    vecIDX = 0

    for pos in range(vec[-1] + 1):
        while vecIDX < len(vec) - 1 and pos > vec[vecIDX]:
            vecIDX += 1
        result.append(vec[vecIDX])

    return result

#calculate weighted log likelihood of test transitions (unnormalized) under train model (normalized)
def averageLL(trainProbs, testProbs):

    ll = 0.0 # start at zero

    weight = 0.0

    for s, nexts in testProbs.items(): #for each item in the test transition 
        trainContext = trainProbs.get(s) #find matching train model probability

        if trainContext is None:
            continue  
                             # if context unseen in train, skip
        for s2, p_test in nexts.items():
            p_train = trainContext.get(s2)

            if p_train is None or p_train <= 0:
                continue  # if transition unseen in train, skip
                             
            ll += p_test * np.log(p_train)
            weight += p_test

    if weight == 0:
        return np.nan
    
    return ll / weight

#### main analysis functions ####

""" general pipeline:

normalize the training set because we are essentially building a smoothed model that we can use to evaluate test set fit
keep test set as count data because we are comparing how well it fits to our smoothed model

 """

##### Lagged Context Markov Sampling--bad performance in synthetic eval, not used #####

# def laggedTransNorm(states, lines, k, smoother, lag):

#     transitions = defaultdict(Counter)
    
#     #for k=0 version, transition matrix turns into simple marginal state distributions
#     if k == 0:
#         for state in states:
#             transitions[()][state] = smoother #add laplace smoother to each curr state

#         for seq in lines:
#             for curr in seq:
#                 transitions[()][curr] += 1 #add one count in corresponding transition cell for each instance of each state

#         total = sum(transitions[()].values())

#         return {(): {s: c / total for s, c in transitions[()].items()}} #normalize

#     for prev in product(states, repeat=k): #add laplace smoother to every combination of prev & curr states

#         for curr in states:
#             transitions[prev][curr] = smoother

#     for seq in lines:
#         seqLength = len(seq)

#         for t in range(k * lag, seqLength):
#             prev = tuple(seq[t - lag * i - 1] for i in reversed(range(k))) # important line: looks back at k previous states each spaced one lag apart, starting from t-1
#             curr = seq[t] # curr state is at time t
#             transitions[prev][curr] += 1 #add 1 count to this prev & curr combination

#     transNorm = {}

#     for cond, counter in transitions.items():
#         total = sum(counter.values())
#         transNorm[cond] = {s: c / total for s, c in counter.items()} #normalize counts by total number of transitions

#     return transNorm

# def laggedTransCount(lines, k, lag):

#     transitions = defaultdict(Counter)

#     # if k is zero, just count instances of each state
#     if k == 0:
#         for seq in lines:
#             for curr in seq:
#                 transitions[()][curr] += 1
#         return transitions
    

#     for seq in lines:
#         seqLength = len(seq)
#         for t in range(k * lag, seqLength):
#             prev = tuple(seq[t - lag * i - 1] for i in reversed(range(k))) # important line: looks back at k previous states each spaced one lag apart, starting from t-1
#             curr = seq[t] # curr state is at time t
#             transitions[prev][curr] += 1  #add 1 count to this prev & curr combination

#     return transitions

##### Collapsed Context Markov Sampling #####

def collapsedTransNorm(states, lines, k, smoother):
    
    transitions = defaultdict(Counter)

    #for k=0 version, transition matrix turns into simple marginal state distributions
    if k == 0:
        for state in states:
            transitions[()][state] = smoother #add laplace smoother to each curr state

        for seq in lines:
            for curr in seq:
                transitions[()][curr] += 1 #add one count in corresponding transition cell for each instance of each state

        total = sum(transitions[()].values())

        return {(): {s: c / total for s, c in transitions[()].items()}} #normalize

    for prev in product(states, repeat=k):

        if hasConsecutiveRepeats(prev): #if prev tuple has consecutive repeats, skip (because this cannot be observed when self-transitions are collapsed)
            continue

        for curr in states:
            transitions[prev][curr] = smoother #add laplace smoother to every valid combination of prev & curr states

    for seq in lines:

        idxs = uniqueIdxs(seq) #find indices of unique states in sequence
        idxsExpanded = expandVec(idxs) #expand indices for mapping

        for t in range(1, len(seq)):

            idxsCollapsed = idxs.index(idxsExpanded[t - 1]) #find corresponding index of each token

            if idxsCollapsed < k - 1:
                continue

            prevIdxsUnique = idxs[idxsCollapsed- (k - 1):idxsCollapsed+ 1]
            prev = tuple(seq[i] for i in prevIdxsUnique)
            curr = seq[t]
            transitions[prev][curr] += 1

    transNorm = {}

    for cond, counter in transitions.items():
        total = sum(counter.values())
        transNorm[cond] = {s: c / total for s, c in counter.items()} #normalize 

    return transNorm

def collapsedTransCount(lines, k):

    transitions = defaultdict(Counter)

    # if k is zero, just count instances of each state
    if k == 0:
        for seq in lines:
            for curr in seq:
                transitions[()][curr] += 1

        return transitions
    
    for seq in lines:
        idxs = uniqueIdxs(seq)
        idxsExpanded = expandVec(idxs)

        for t in range(1, len(seq)):
            idxsCollapsed= idxs.index(idxsExpanded[t - 1])

            if idxsCollapsed< k - 1:
                continue

            prevIdxsUnique = idxs[idxsCollapsed- (k - 1):idxsCollapsed+ 1]
            prev = tuple(seq[i] for i in prevIdxsUnique)
            curr = seq[t]
            transitions[prev][curr] += 1

    return transitions

##### comparative analysis #####

def fitModel(species, i, trainLines, testLines, kMax, smootherSet):
 
    results = []

    #find unique states in corpus
    states = sorted({char for line in trainLines for char in line})

    for k in range(0, kMax + 1):

        for smoother in smootherSet:

            train_trans = collapsedTransNorm(states = states, lines=trainLines, k=k, smoother=smoother)

            test_trans  = collapsedTransCount(lines=testLines, k=k)

            ll = averageLL(train_trans, test_trans)

            results.append({
                'species': species, 
                'k': k,
                'sim': i,
                'smoother': smoother,
                'log_likelihood': ll,
                'num_train_states': len(train_trans),
                'num_test_states':  len(test_trans),
            })
            print(f"{species} sim {i}: k={k}")

    return pd.DataFrame(results)


In [ ]:
# DO IT! #

dat = pd.read_csv("fish-data/raw/PrettyDat.csv")

# params to test
nIter = 100
kmax=6
smoothers=[0.001, 0.01, 0.1]

# run iterations
results = []

for species, species_df in dat.groupby("species"):

    sequences = readSequences(species_df)

    for i in range(1, nIter+1):

        train_lines, test_lines = train_test_split(
            sequences,
            test_size=0.2)
        
        iModel = fitModel(species = species, i = i, trainLines=train_lines, testLines=test_lines, kMax = kmax, smootherSet = smoothers)

        results.append(iModel)


results_df = pd.concat(results, ignore_index=True)
results_df.to_csv("fish-data/fishResults.csv", index=False)
        

Species bre, k=0, smoother=0.001: LL=-2.2406
Species bre, k=0, smoother=0.01: LL=-2.2406
Species bre, k=0, smoother=0.1: LL=-2.2406
Species bre, k=1, smoother=0.001: LL=-0.0694
Species bre, k=1, smoother=0.01: LL=-0.0694
Species bre, k=1, smoother=0.1: LL=-0.0695
Species bre, k=2, smoother=0.001: LL=-0.0791
Species bre, k=2, smoother=0.01: LL=-0.0787
Species bre, k=2, smoother=0.1: LL=-0.0785
Species bre, k=3, smoother=0.001: LL=-0.0964
Species bre, k=3, smoother=0.01: LL=-0.0940
Species bre, k=3, smoother=0.1: LL=-0.0935
Species bre, k=4, smoother=0.001: LL=-0.2562
Species bre, k=4, smoother=0.01: LL=-0.2517
Species bre, k=4, smoother=0.1: LL=-0.2525
Species bre, k=5, smoother=0.001: LL=-0.8216
Species bre, k=5, smoother=0.01: LL=-0.7816
Species bre, k=5, smoother=0.1: LL=-0.7555
Species bre, k=6, smoother=0.001: LL=-1.0775
Species bre, k=6, smoother=0.01: LL=-1.0735
Species bre, k=6, smoother=0.1: LL=-1.0760
Species bre, k=0, smoother=0.001: LL=-2.1297
Species bre, k=0, smoother=0.01